# Calculate first latency pdf- probability density function of the time to first opening

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import math


In [ ]:
from scalcs.samples import samples
from scalcs import firstlatency as fl
from scalcs import qmatlib as qm


##### Load Colquhoun & Hawkes 1982 numerical example

In [ ]:
c0 = 0.0 # 0 mM
c1 = 0.0001 # 0.1 mM
mec0 = samples.CH82()
mec0.set_eff('c', c0)
mec1 = samples.CH82()
mec1.set_eff('c', c1)

##### Helper functions are now in `scalcs.firstlatency`
Ideal, asymptotic and exact first-latency pdfs are implemented in
`scalcs/firstlatency.py` and imported above as `fl`.


In [ ]:
# Helper functions (asymptotic_areas_first_latency, asymptotic_pdf,
# exact_GAMAxx) have been moved to scalcs/firstlatency.py.
# Use fl.asymptotic_roots, fl.asymptotic_areas, fl.asymptotic_pdf,
# fl.gamma_coefficients and fl.exact_pdf below.


#### The special case of a simple step from zero concentration

In [ ]:
points = 512
tstart = 1.0e-6 
tend = 0.5e-3
tseq = np.logspace(math.log10(tstart), math.log10(tend), points)

In [ ]:
tres = 0.0001  # 0.1 ms
phi_shut = qm.pinf(mec0.Q)[mec0.kA:]


In [ ]:
# Ideal first-latency pdf (no missed events)
eigs, areas_ideal = fl.ideal_components(mec1.QFF, phi_shut)
ipdf = fl.ideal_pdf(tseq, mec1.QFF, phi_shut)
print("Ideal: {} components".format(mec1.kF))
print("  eigenvalues (1/s): {}".format(eigs))
print("  areas (sum={:.6f}): {}".format(areas_ideal.sum(), areas_ideal))


In [ ]:
# Asymptotic first-latency pdf (HJC correction for missed events)
roots = fl.asymptotic_roots(tres, mec1)
areas_asym = fl.asymptotic_areas(tres, roots, phi_shut, mec1)
tau = -1.0 / roots
apdf = fl.asymptotic_pdf(tseq, tres, tau, areas_asym)
print("Asymptotic roots: {}".format(roots))
print("Asymptotic areas (sum={:.6f}): {}".format(areas_asym.sum(), areas_asym))


In [ ]:
# Exact first-latency pdf (HJC: tres <= t <= 3*tres corrected, asymptotic beyond)
eigvals, g00, g10, g11 = fl.gamma_coefficients(tres, phi_shut, mec1)
epdf = fl.exact_pdf(tseq, tres, roots, areas_asym, eigvals, g00, g10, g11)


In [ ]:
plt.plot(tseq, apdf, 'b--');
plt.plot(tseq, epdf, 'b-');
plt.plot(tseq, ipdf, 'r--');
#plt.xlim([0.0, 0.00004]);
print('red dashed- ideal pdf; blue dashed- asymptotic pdf; blue solid- exact + asymptotic pdf')